In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## Data Loading

In [ ]:
train_dir = "/kaggle/input/asl-dataset/asl_sample_train1"
val_dir = "/kaggle/input/asl-dataset/asl_sample_train3"
img_size = (224, 224)   # EfficientNetB0 input size
batch_size = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"  
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical"
)


## Create Classes into map 

In [ ]:
num_classes = len(train_ds.class_names)
print("Classes:", train_ds.class_names)
print("Number of classes:", num_classes)

## Normalization

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))


## Model building

In [ ]:
base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224,224,3))
base_model.trainable = False   

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)  
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)


In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Training (Feature Extraction)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

## Fine Tunning

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:  
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5),  
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

## Evaluation

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"Validation Accuracy: {acc*100:.2f}%")